# Inferencia REST contra `model_serving`

Este notebook consume la API FastAPI definida en `../model_serving/main.py` usando la librería `requests`. Se cubren los dos endpoints:

1. **`POST /upload_model/`** — sube los tres `.pkl` (preprocesador, filtro y calibrador Venn-Abers) que produjo el notebook `12_complete_pipeline.ipynb`.
2. **`POST /predict/`** — envía un crédito de `data/df_test_small.csv` como JSON y recibe `p_0`, `p`, `p_1`.

**Pre-requisito**: el servidor debe estar corriendo. Desde `../model_serving/`:

```bash
uv run uvicorn main:app --reload
```

## 1 — Subida de los pickles vía `/upload_model/`

Mandamos los tres artefactos como `multipart/form-data`. Los nombres de campo (`preprocessing`, `filtering`, `model`) deben coincidir exactamente con los parámetros del endpoint.

In [1]:
import requests

API_URL = "http://127.0.0.1:8000"

In [ ]:


with open("preprocessor.pkl", "rb") as f_pre, \
     open("filter.pkl", "rb") as f_filt, \
     open("va_cal.pkl", "rb") as f_model:
    files = {
        "preprocessing": ("preprocessing.pkl", f_pre, "application/octet-stream"),
        "filtering":     ("filtering.pkl",     f_filt, "application/octet-stream"),
        "model":         ("model.pkl",         f_model, "application/octet-stream"),
    }
    resp = requests.post(f"{API_URL}/upload_model/", files=files)

resp.raise_for_status()
resp.json()

## 2 — Predicción de un ejemplo de test vía `/predict/`

Leemos `data/df_test_small.csv`, tomamos una fila como si fuera una nueva solicitud y la enviamos como JSON (un único dict). La API responde con `p_0`, `p` y `p_1`.

In [ ]:
import pandas as pd

df_test = pd.read_csv("data/df_test_small.csv")
ejemplo = df_test[df_test["desc"].notna()].iloc[0].to_dict()

resp = requests.post(f"{API_URL}/predict/", json=ejemplo)
resp.raise_for_status()
resp.json()